- orbital area: eye area 
- nose bulge: nose area
- cheek bulge: cheek area
- ear position: ear area
- whisker change: side bart area

## Task Description
- Why I use DLC?
    - First I mark the muzzle area by myself, then I train DLC, so that DLC can mark muzzle area for me. At end I can use the marked muzzle area to crop image.

- How do you want to train DLC?
   - there are 4 kinds of mice, each mouse has different number of images. and only 3500 images are labeled.
   - I plan to first filter out those labeled images.
   - Then I want to group images by mice type.
   - Then I want to use same proportion of images. E.g. if there are 80% of brown mice in total, then I also wish to have 80% brown mice in my human marked images for DLC to train.
   - I plan to only mark the eye-left, eye-right and nose-tip area of mice. Because with this triangle area, I can already crop a rectangle muzzle area.

The file 00_full_image_classification.ipynb is what I already coded. Now I want to do the following thing.

ToDo:  
The images are stored in the path mouse_dataset/images.  
E.g. the image 000001.jpg can be found unter path mouse_dataset/images/000001.jpg.  

The file MouseGrimaceFaces_main.csv can be found unter path mouse_dataset/MouseGrimaceFaces_main.csv.  
The file MouseGrimaceFaces_mgs.csv can be found unter path mouse_dataset/MouseGrimaceFaces_mgs.csv.

Since there are around 35000 images but only around 3500 images are labeled with mgs score. I wish to filter out those images which has mgs score, and store those images unter the path mouse_dataset/images_mgs/   

if the mgs score is 9 or NA, or value missing, it means it is not usable. I don't need such images.



Question: if mgs is a mix of 0,1,2 and 9, should I keep it or remove it???

In [ ]:
print("hello")

In [1]:
from pathlib import Path
import shutil
import pandas as pd

# -------------------------
# Paths
# -------------------------
DATASET_DIR = Path("mouse_dataset")

IMAGE_DIR = DATASET_DIR / "images"
MGS_CSV = DATASET_DIR / "MouseGrimaceFaces_mgs.csv"
OUTPUT_DIR = DATASET_DIR / "images_mgs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Load MGS csv
# -------------------------
mgs_df = pd.read_csv(MGS_CSV)

print("Original MGS rows:", len(mgs_df))
print(mgs_df.head())

# -------------------------
# Score columns
# -------------------------
# columns like ot1, nb1, cb1, ep1, wc1, ...
score_cols = [
    col for col in mgs_df.columns
    if col not in ["index", "subset"]
]

# Convert scores to numeric:
# valid scores: 0, 1, 2
# invalid: 9
# missing/no score: "-", NaN, empty cells
scores = mgs_df[score_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# -------------------------
# Filter usable images
# 至少有一个有效分数(0,1,2)，不能出现 9
# -------------------------
# Condition 1: at least one valid score 0/1/2 
has_valid_score = scores.isin([0, 1, 2]).any(axis=1)

# Condition 2: no score equals 9
has_no_9 = ~scores.eq(9).any(axis=1)

usable_df = mgs_df[has_valid_score & has_no_9].copy()

print("Usable MGS rows:", len(usable_df))
print("Removed rows:", len(mgs_df) - len(usable_df))

# -------------------------
# Copy usable images
# -------------------------
copied = 0
missing_files = []

for filename in usable_df["index"]:
    src = IMAGE_DIR / filename
    dst = OUTPUT_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing_files.append(filename)

print("Copied images:", copied)
print("Missing image files:", len(missing_files))

if missing_files:
    print("First missing files:")
    print(missing_files[:10])

# Optional: save list of usable images
usable_csv_path = DATASET_DIR / "usable_mgs_images.csv"
usable_df.to_csv(usable_csv_path, index=False)

print("Saved usable image list to:", usable_csv_path)
print("Copied images stored under:", OUTPUT_DIR)

Original MGS rows: 3406
        index  ot1  nb1  cb1  ep1  wc1 ot2 nb2 cb2 ep2  ... nb11 cb11 ep11  \
0  000008.jpg    -    -    -    -    -   -   -   -   -  ...    1    0    0   
1  000009.jpg    1    1    0    1    2   0   0   0   0  ...    -    -    -   
2  000015.jpg  NaN  NaN  NaN  NaN  NaN   -   -   -   -  ...  NaN  NaN  NaN   
3  000027.jpg  NaN  NaN  NaN  NaN  NaN   1   1   1   1  ...  NaN  NaN  NaN   
4  000031.jpg    -    -    -    -    -   -   -   -   -  ...    1    0    0   

  wc11 ot12 nb12 cb12 ep12 wc12 subset  
0    0    0    1    0    1    1     KH  
1    -    -    -    -    -    -     KH  
2  NaN  NaN  NaN  NaN  NaN  NaN     JW  
3  NaN  NaN  NaN  NaN  NaN  NaN     JW  
4    0    0    1    1    1    1     KH  

[5 rows x 57 columns]
Usable MGS rows: 2163
Removed rows: 1243
Copied images: 2163
Missing image files: 0
Saved usable image list to: mouse_dataset/usable_mgs_images.csv
Copied images stored under: mouse_dataset/images_mgs


## DLC Training
Now you need to select images and train DLC.  
There are 2163 usable images.  
I plan to mark 250 images by myself and try out the effect.  
I plan to use those body parts. 
- nose
- eye_left
- eye_right

- conda activate DEEPLABCUT  
- python -m deeplabcut


apply training result on our images

In [1]:
import deeplabcut

config_path = "mouse-mgs-2026-06-04/config.yaml"

deeplabcut.analyze_time_lapse_frames(
    config_path,
    directory="mouse_dataset/images_mgs",
    frametype=".jpg",
    save_as_csv=True
)

Loading DLC 3.0.0rc13...


100%|██████████| 2163/2163 [41:50<00:00,  1.16s/it] 


Saving predictions to mouse_dataset/images_mgs/image_predictions_DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200.h5
Saving CSV as mouse_dataset/images_mgs/image_predictions_DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200.h5


{'mouse_dataset/images_mgs/015563.jpg': {'bodyparts': array([[[4.91439026e+02, 1.05203943e+03, 6.51453614e-01],
          [5.15586731e+02, 8.87096069e+02, 1.10905886e-01],
          [6.80192993e+02, 7.17847656e+02, 8.67178500e-01]]], dtype=float32)},
 'mouse_dataset/images_mgs/005002.jpg': {'bodyparts': array([[[1.2114812e+02, 3.5543195e+02, 7.0777595e-01],
          [1.2307438e+02, 3.5344354e+02, 4.8014559e-02],
          [2.0749226e+02, 2.4147412e+02, 1.0000000e+00]]], dtype=float32)},
 'mouse_dataset/images_mgs/007173.jpg': {'bodyparts': array([[[8.4469385e+02, 5.8516272e+02, 4.9721959e-01],
          [5.6830634e+02, 5.1832043e+02, 6.1422360e-01],
          [4.6649792e+02, 9.8802826e+02, 2.4397537e-01]]], dtype=float32)},
 'mouse_dataset/images_mgs/014669.jpg': {'bodyparts': array([[[2.5101691e+02, 2.5010857e+02, 7.4040079e-01],
          [1.7473895e+02, 1.9027591e+02, 7.6051390e-01],
          [1.7524144e+02, 1.9052127e+02, 7.4994683e-02]]], dtype=float32)},
 'mouse_dataset/images_

read the h5 file of DLC after the step above

In [2]:
import pandas as pd
from pathlib import Path

dlc_h5_path = Path(
    "mouse_dataset/images_mgs/image_predictions_DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200.h5"
)

df = pd.read_hdf(dlc_h5_path)

print(df.shape)
df.head()

(2163, 9)


scorer                              DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200  \
individuals                                                                   animal   
bodyparts                                                                       nose   
coords                                                                             x   
mouse_dataset/images_mgs/015563.jpg                                       491.439026   
mouse_dataset/images_mgs/005002.jpg                                       121.148117   
mouse_dataset/images_mgs/007173.jpg                                       844.693848   
mouse_dataset/images_mgs/014669.jpg                                       251.016907   
mouse_dataset/images_mgs/011739.jpg                                       136.882904   

scorer                                                                   \
individuals                                                               
bodyparts                                                      left_eye   
coords                                         y likelihood           x   
mouse_dataset/images_mgs/015563.jpg  1052.039429   0.651454  515.586731   
mouse_dataset/images_mgs/005002.jpg   355.431946   0.707776  123.074379   
mouse_dataset/images_mgs/007173.jpg   585.162720   0.497220  568.306335   
mouse_dataset/images_mgs/014669.jpg   250.108566   0.740401  174.738953   
mouse_dataset/images_mgs/011739.jpg   217.997849   0.501825  111.624512   

scorer                                                                  \
individuals                                                              
bodyparts                                                    right_eye   
coords                                        y likelihood           x   
mouse_dataset/images_mgs/015563.jpg  887.096069   0.110906  680.192993   
mouse_dataset/images_mgs/005002.jpg  353.443542   0.048015  207.492264   
mouse_dataset/images_mgs/007173.jpg  518.320435   0.614224  466.497925   
mouse_dataset/images_mgs/014669.jpg  190.275909   0.760514  175.241440   
mouse_dataset/images_mgs/011739.jpg  155.685181   0.245113  112.576607   

scorer                                                      
individuals                                                 
bodyparts                                                   
coords                                        y likelihood  
mouse_dataset/images_mgs/015563.jpg  717.847656   0.867178  
mouse_dataset/images_mgs/005002.jpg  241.474121   1.000000  
mouse_dataset/images_mgs/007173.jpg  988.028259   0.243975  
mouse_dataset/images_mgs/014669.jpg  190.521271   0.074995  
mouse_dataset/images_mgs/011739.jpg  155.782944   0.301734

In [3]:
print(df.columns)

MultiIndex([('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...),
            ('DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200', ...)],
           names=['scorer', 'individuals', 'bodyparts', 'coords'])


In [4]:
scorer = df.columns.get_level_values(0)[0]

bodyparts = df.columns.get_level_values(1).unique()
print("scorer:", scorer)
print("bodyparts:")
for bp in bodyparts:
    print(bp)

scorer: DLC_Resnet50_mouseJun4shuffle1_snapshot_best-200
bodyparts:
animal


In [ ]:
NOSE = "nose-tip"
EYE_L = "left_eye"
EYE_R = "right_eye"